## Imports

In [37]:
import sklearn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib as mpl
import scipy.io as sio
import itertools

In [8]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


## Linear Autoencoders

In [23]:
class SCA_encoder(nn.Module):
    '''
    Input: a (T * N) matrix X, of N neurons recorded over T timesteps.
    Output: an affine mapping of X to K-dimensional space.
    '''
    def __init__(
        self,
        N: 100, # number of neurons
        K: 3 # dimensionality
    ):
        super().__init__()
        self.U = nn.Linear(N,K)

    def forward(self,x):
        encoded_x = self.U(x)
        return encoded_x

class SCA_decoder(nn.Module):
    '''
    Input: affine mapped X in K-dimensional space
    Output: X_hat, or the reconstructed X back in neural activity space
    '''
    def __init__(
        self,
        N: 100, # number of neurons
        K: 3 # dimensionality
    ):
        super().__init__()
        self.V = nn.Linear(K,N)

    def forward(self, encoded_x):
        normalized_V_weight = nn.functional.normalize(self.V.weight.data,dim=1)
        self.V.weight.data = normalized_V_weight
        x_hat = self.V(encoded_x)
        return x_hat
        

## Training Function

In [45]:
# Have not yet implemented sample-weighting
def train(X, encoder, decoder, optimizer, lambda_sparse = 0, lambda_orth = 0, sample_weighting=False):

    if sample_weighting:
        print("not yet!")
        return None

    X = X.to(device)

    # Calculate reconstruction loss
    encoded_X = encoder(X)
    decoded_X = decoder(encoded_X)
    reconstruction_loss = torch.norm(X - decoded_X)**2

    # Calculate sparsity penalty
    sparsity_loss = lambda_sparse * torch.norm(encoded_X)

    # Calculate orthogonality penalty
    V_weight = nn.functional.normalize(decoder.V.weight.data,dim=1)
    VTV = torch.matmul(V_weight.transpose(1,0),V_weight)
    I = torch.eye(VTV.size(dim=0),device=device)
    orth_loss = lambda_orth * torch.norm(VTV - I)**2

    # Sum to calculate loss
    loss = reconstruction_loss + sparsity_loss + orth_loss

    # Backpropagate

    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    return loss.item()


## Training Loop

In [51]:
epochs = 100

def training_loop(X, encoder, decoder, optimizer, lambda_sparse = 0, lambda_orth = 0, sample_weighting=False, epochs=10):
    if sample_weighting:
        print("not yet!")
        return None
    losses = torch.zeros(epochs)
    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")
        loss_t = train(X, encoder, decoder, optimizer, lambda_sparse, lambda_orth, sample_weighting)
        print(f"loss: {loss_t:>7f}  [{t:>5d}/{epochs:>5d}]")
        losses[t] = loss_t
    
    return losses

In [54]:
N = 10
K = 3
T = 15

encoder_ex = SCA_encoder(N=N,K=K).to(device)
decoder_ex = SCA_decoder(N=N,K=K).to(device)

all_params = itertools.chain(encoder_ex.parameters(), decoder_ex.parameters())
optimizer_ex = torch.optim.Adam(all_params)
data_ex = torch.randn(T,N)

training_loop(data_ex,encoder_ex,decoder_ex,optimizer_ex,lambda_sparse=0.1,lambda_orth=0.1,epochs=epochs)

Epoch 1
-------------------------------
loss: 233.687988  [    0/  100]
Epoch 2
-------------------------------
loss: 232.251328  [    1/  100]
Epoch 3
-------------------------------
loss: 230.826523  [    2/  100]
Epoch 4
-------------------------------
loss: 229.414062  [    3/  100]
Epoch 5
-------------------------------
loss: 228.014557  [    4/  100]
Epoch 6
-------------------------------
loss: 226.628586  [    5/  100]
Epoch 7
-------------------------------
loss: 225.256592  [    6/  100]
Epoch 8
-------------------------------
loss: 223.898178  [    7/  100]
Epoch 9
-------------------------------
loss: 222.552505  [    8/  100]
Epoch 10
-------------------------------
loss: 221.219055  [    9/  100]
Epoch 11
-------------------------------
loss: 219.897278  [   10/  100]
Epoch 12
-------------------------------
loss: 218.586761  [   11/  100]
Epoch 13
-------------------------------
loss: 217.287445  [   12/  100]
Epoch 14
-------------------------------
loss: 215.999176  [

tensor([233.6880, 232.2513, 230.8265, 229.4141, 228.0146, 226.6286, 225.2566,
        223.8982, 222.5525, 221.2191, 219.8973, 218.5868, 217.2874, 215.9992,
        214.7220, 213.4559, 212.2010, 210.9575, 209.7252, 208.5043, 207.2947,
        206.0966, 204.9098, 203.7345, 202.5706, 201.4180, 200.2768, 199.1470,
        198.0286, 196.9216, 195.8259, 194.7415, 193.6684, 192.6066, 191.5560,
        190.5166, 189.4883, 188.4711, 187.4649, 186.4696, 185.4852, 184.5116,
        183.5488, 182.5966, 181.6550, 180.7238, 179.8030, 178.8926, 177.9923,
        177.1022, 176.2221, 175.3519, 174.4916, 173.6410, 172.8000, 171.9686,
        171.1467, 170.3341, 169.5307, 168.7365, 167.9513, 167.1750, 166.4076,
        165.6490, 164.8990, 164.1575, 163.4245, 162.6998, 161.9833, 161.2750,
        160.5747, 159.8823, 159.1978, 158.5210, 157.8518, 157.1902, 156.5361,
        155.8892, 155.2496, 154.6172, 153.9918, 153.3734, 152.7619, 152.1571,
        151.5590, 150.9675, 150.3824, 149.8038, 149.2316, 148.66